# 01 - INE Mobility Data Ingestion & Cleaning

Upload, cleaning and preparation of INE's dataset (Instituto Nacional de Estadística).

## 1. Imports

In [1]:
import pandas as pd
import glob
from datetime import datetime
from unidecode import unidecode
from pathlib import Path
import numpy as np

## 2. Path configuration

In [2]:
RAW_DATA_PATH = Path("../data/raw/MAIN_INE_mobility_data")
INTERIM_PATH = Path("../data/interim")
INTERIM_PATH.mkdir(
    parents=True,
    exist_ok=True
)


## 3. Functions to load, process & concat Excels

In [3]:
def get_excel_files(folder):
    """
    Searches for all Excel files (.xlsx) within a specified directory.
    Args: folder (str): The directory path to search in.
    Returns: list: A list of file paths for the discovered Excel files.
    Raises: ValueError: If no Excel files are found in the directory.
    """
    
    path_pattern = f"{folder}/*.xlsx"
    files = glob.glob(path_pattern)

    if not files:
        raise ValueError("Excels not found")
    
    print(f"✔ {len(files)} docs found")
    return files

In [4]:
def extract_excel_sheets(file_path, verbose=False):
    """
    Extracts all relevant sheets from an INE Excel file and appends metadata.
    
    Skips the first sheet (an overview) and adds 'source_file' 
    and 'sheet' columns to track the origin of the data.

    Args:
        file_path (str): The system path to the Excel file.
        verbose (bool, optional): If True, prints the name of each processed sheet. Defaults to False.

    Returns:
        list: A list of pandas DataFrames, one for each parsed sheet.
    """
    lista_sheets_df = []
    
    try:
        xls = pd.ExcelFile(file_path)

        for sheet_name in xls.sheet_names[1:]:# Sheet 0 is always metadada
            df = xls.parse(sheet_name)

            df["source_file"] = file_path
            df["sheet"] = sheet_name

            lista_sheets_df.append(df)

            if verbose:
                print(f"✔ {sheet_name}")

    except Exception as e:
        print(f"! Error with {file_path}: {e}")

    return lista_sheets_df

In [5]:
def process_all_files(files):
    """
    Iterates over a list of Excel files and extracts their sheets.
    Args: files (list): A list of file paths to process.
    Returns: list: A flattened list of pandas DataFrames, containing all extracted sheets.
    """
    all_dfs = []

    for file in files:
        sheets = extract_excel_sheets(file)
        all_dfs.extend(sheets)

    return all_dfs

In [6]:
def concat_dataframes(df_list):
    """
    Concatenates a list of DataFrames into a single DataFrame.
    Args: df_list (list): A list containing pandas DataFrames to concatenate.
    Returns: df: A single, combined DataFrame.
    Raises: ValueError: If the input list is empty.
    """
    if not df_list:
        raise ValueError("The list of DataFrames is empty")

    return pd.concat(df_list, ignore_index=True)

## 4. Cleaning functions

In [7]:
def normalize_columns(df):
    """
    Standardizes DataFrame column names for easier querying.
    
    Converts all column names to lowercase, removes leading/trailing 
    whitespaces, and replaces internal spaces with underscores.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A new DataFrame with normalized column names.
    """

    df = df.copy()

    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_", regex=False)
    )

    return df

In [8]:
def rename_columns(df):
    """
    Renames specific Spanish column names to standardized English counterparts.
    Args: df with raw column names.
    Returns: df with renamed columns.
    """

    df = df.copy()
    df = df.rename(columns={

        "mes": "period",

        "mun_orig_cod": "depart_city_code",
        "mun_orig": "depart_city",

        "prov_orig_cod": "depart_province_code",
        "prov_orig": "depart_province",

        "destino": "destination",
        "destino_cod": "destination_code",

        "turistas": "total_tourists"
    })

    return df

In [9]:
def convert_types(df):
    """
    Converts temporal columns to appropriate datetime and numeric formats.
    
    Extracts 'year' and 'month' into separate columns. Coerces parsing 
    errors into NaT (Not a Time) and flags if any nulls are generated.

    Args:
        df: The DataFrame containing a 'period' column.

    Returns:
        df2: The DataFrame with updated data types.
    """

def convert_types(df):
    df = df.copy()

    df["period"] = pd.to_datetime(df["period"], errors="coerce")
    
    nulls = df["period"].isnull().sum()
    if nulls > 0:
        print(f"⚠ Warning: {nulls} rows with unparseable dates → NaT")
        display(df[df["period"].isnull()].head())
    
    df["year"] = df["period"].dt.year
    df["month"] = df["period"].dt.month

    return df

In [10]:
def classify_destination_type(df):
    """
    Classifies destinations into 'total', 'region', or 'country' based on codes.
    
    Args: df : The DataFrame containing 'destination' and 'destination_code' columns.
    Returns: df: The DataFrame with a new 'destination_type' categorical column.
    """
    
    df = df.copy()

    conditions = [
        df["destination"].str.lower() == "total",
        df["destination_code"] < 100
    ]

    choices = [
        "total",
        "region"
    ]

    df["destination_type"] = np.select(
        conditions,
        choices,
        default="country"
    )

    return df

In [11]:
def extract_countries_master(df):
    """
    Extracts a unique list of countries and their corresponding codes 
    Args: df: The fully cleaned INE dataset.
    Returns: a df serving as a country lookup table.
    """
    
    df_countries = (
        df[df["destination_type"] == "country"]
        [["destination", "destination_code"]]
        .drop_duplicates(subset=["destination_code"])
        .rename(columns={
            "destination": "country",
            "destination_code": "country_code"
        })
    )
    return df_countries

## 5. Quality assurance

In [12]:
def data_quality_check(df):
    """
    Performs a quality assurance (QA) check on the DataFrame.
    
    Prints a report evaluating exact duplicates, logical duplicates (same route 
    and month), negative anomalies in tourist counts and missing values.

    Args:
        df: The cleaned DataFrame to be evaluated.

    Returns:
        None: This function only prints to the standard output.
    """
    
    print("\n--- EXACT DUPLICATES ---")
    duplicates = df.duplicated().sum()
    print(f"Total: {duplicates}")
    
    print("\n--- LOGICAL DUPLICATES ---")
    # Check for more than one record for the same origin, destination, and month
    logical_duplicates = df.duplicated(subset=['period', 'depart_city_code', 'destination_code']).sum()
    print(f"Same route and month: {logical_duplicates}")

    print("\n--- NEGATIVE VALUES ---")
    # Check if there are any negative values in the tourists column
    neg_tourists = (df["total_tourists"] < 0).sum()
    print(f"Negative tourist records: {neg_tourists}")

    print("\n--- NULLS ---")
    print(df.isnull().sum())

## 6. Main functions

In [13]:
def main_load():
    """
    Orchestrates the entire raw data loading process: reads Excel files, extracts sheets 
    and concatenates them into a single raw DataFrame.

    Returns: df with the concatenated raw dataset.
    """
    print(f"Loading Excel files from {RAW_DATA_PATH}")
    files = get_excel_files(RAW_DATA_PATH)
    dfs = process_all_files(files)
    df = concat_dataframes(dfs)

    return df

In [14]:
def main_clean(df):
    """
    Executes the main data cleaning pipeline for the raw INE dataset.
    
    Applies a sequential chain of transformations: column normalization, 
    renaming to English variables, type casting and destination classification.

    Args:
        df: The raw, concatenated INE pandas DataFrame.

    Returns:
        df: The fully cleaned and standardized DataFrame.
    """
    df = normalize_columns(df)
    df = rename_columns(df)
    df = convert_types(df)
    df = classify_destination_type(df)
    return df

## 7. Main pipeline

In [15]:
df_raw = main_load()
df_clean = main_clean(df_raw)
data_quality_check(df_clean)

df_countries_master = extract_countries_master(df_clean)

df_clean.head()

Loading Excel files from ..\data\raw\MAIN_INE_mobility_data
✔ 8 docs found

--- EXACT DUPLICATES ---
Total: 0

--- LOGICAL DUPLICATES ---
Same route and month: 0

--- NEGATIVE VALUES ---
Negative tourist records: 0

--- NULLS ---
period                   0
depart_city_code         0
depart_city             33
destination_code         0
destination              0
total_tourists           0
depart_province_code     0
depart_province          0
source_file              0
sheet                    0
year                     0
month                    0
destination_type         0
dtype: int64


,period,depart_city_code,depart_city,destination_code,destination,total_tourists,depart_province_code,depart_province,source_file,sheet,year,month,destination_type
0,2019-07-01,1001,Alegría-Dulantzi,0,Total,248,1,Araba/Álava,..\data\raw\MAIN_INE_mobility_data\exp_tmov_em...,2019-07,2019,7,total
1,2019-07-01,1001,Alegría-Dulantzi,10,Europa,224,1,Araba/Álava,..\data\raw\MAIN_INE_mobility_data\exp_tmov_em...,2019-07,2019,7,region
2,2019-07-01,1001,Alegría-Dulantzi,11,Unión Europea,190,1,Araba/Álava,..\data\raw\MAIN_INE_mobility_data\exp_tmov_em...,2019-07,2019,7,region
3,2019-07-01,1001,Alegría-Dulantzi,110,Francia,77,1,Araba/Álava,..\data\raw\MAIN_INE_mobility_data\exp_tmov_em...,2019-07,2019,7,country
4,2019-07-01,1001,Alegría-Dulantzi,126,Alemania,40,1,Araba/Álava,..\data\raw\MAIN_INE_mobility_data\exp_tmov_em...,2019-07,2019,7,country


### 7.1. Quality Assurance Deep Dive

In [16]:
# Check Logical Duplicates in Main Dataset

logical_dupes = df_clean[
    df_clean.duplicated(subset=['period', 'depart_city_code', 'destination_code'], keep=False)
]

if not logical_dupes.empty:
    print("[WARNING] Displaying logical duplicates for manual review:")
    display(logical_dupes.sort_values(by=['period', 'depart_city_code', 'destination_code']).head(10))
else:
    print("✔ No logical duplicates found. Safe to proceed.")

print("-" * 60)

# Investigate Country Naming Conflicts
# Checks if INE used different string names for the same country code
conflicts = df_clean.groupby("destination_code")["destination"].nunique()
suspicious_codes = conflicts[conflicts > 1].index

if len(suspicious_codes) > 0:
    print(f"[WARNING] Found {len(suspicious_codes)} country codes with multiple string names:")
    display(df_clean[df_clean["destination_code"].isin(suspicious_codes)][["destination_code", "destination"]].drop_duplicates())
else:
    print("✔ Country names are 100% consistent per code. Safe to proceed.")

✔ No logical duplicates found. Safe to proceed.
------------------------------------------------------------
✔ Country names are 100% consistent per code. Safe to proceed.


## 8. Dataset export

In [17]:
# Export main dataset
df_clean.to_parquet(
    INTERIM_PATH / "01_ine_clean_v2.parquet", index=False
)

# Export countries master
df_countries_master.to_parquet(
    INTERIM_PATH / "01_countries_ine_master_v2.parquet", index=False
)

In [18]:
#Export countries to check manually
df_countries_master.to_excel(
    INTERIM_PATH / "01_unique_countries_ine.xlsx", index=False
)

In [19]:
df_countries_master.sample(10)

,country,country_code
23004,Nueva Zelanda,504
23603,Bosnia y Herzegovina,145
2487,Estonia,141
35018,Jamaica,322
127,Andorra,124
2495,Serbia,157
2523,Ecuador,345
11490,Iraq,412
21294,Liechtenstein,116
97783,Ghana,217


## <font color="red">IMPORTANT:</font> Dataset Limitations

This notebook ingests INE's experimental tourism mobility dataset. The following
limitations are known and must be considered when interpreting downstream results:

**1. Statistical suppression of small municipalities (threshold: < 30 tourists)**
INE suppresses all origin-destination combinations with fewer than 30 tourists 
per month. 
Additionally, combinations derivable by subtraction from published totals are 
also suppressed. This can create a systematic geographic bias toward large urban 
centers. Rural and low-population provinces are structurally underrepresented.
Consequence: territorial concentration findings may partially reflect data suppression,
not only true demand concentration.

**2. SIM card positioning methodology**
The dataset tracks mobile device presence, not verified tourist activity. Business
travelers, seasonal residents, and repeat travelers are mixed in with leisure tourists.
The "outbound tourism" metric is technically "outbound mobile device presence abroad".

**3. Children excluded**
Travelers under a certain age threshold are excluded from the dataset by construction.
This underestimates family travel demand.

**4. eSIM, local SIM, and airplane mode**
The methodology captures only devices connected to Spanish operators' roaming 
networks. Travelers using the following may be totally or partially absent from the dataset:
- Local destination SIMs (disconnect from Spanish operator network)
- eSIMs configured without roaming
- Airplane mode throughout the trip
This likely undercounts cost-conscious travelers and long-haul travelers (who more frequently use local SIMs).

**5. Trip-based counting, not person-based**
The INE dataset counts trips, not unique travelers, although the column is labelled `total_tourists` throughout this project. A resident who travels abroad three times in the same month is counted as three tourists.
This inflates demand figures for high-frequency destinations (e.g. Portugal, France,
Andorra) relative to long-haul destinations where repeat travel within a month is rare.

**Partial mitigation applied in this project:**
This analysis mainly focuses on non-European destinations, which substantially 
reduces this bias. Repeat travel to long-haul destinations within a single month is 
logistically and economically rare for most travelers. The trip-to-person 
ratio for extra-European destinations can be reasonably approximated as 1:1.

Residual exception: Morocco and other North African destinations may still exhibit 
some repeat-travel inflation, particularly among Spanish residents of North African 
origin. This will be considered when interpreting North Africa figures.

**6. Market share correction factors**
Raw data from the three operators covers approximately 75% of Spanish mobile users.
INE applies quarterly correction factors (based on CNMC market share data) to 
extrapolate to the total population. These factors vary by country of destination and by
number of operators providing data for each country.
Consequence: estimates for niche or low-volume destinations (where fewer operators 
report data) carry higher uncertainty than estimates for major destinations.

**7. Provincial data is estimated, not directly observed**
Province-level figures are not directly measured. INE calculates them by:
1. Computing each province's share of national trips per destination country (per operator)
2. Averaging the three operators' shares
3. Applying adjusted shares to national totals.
Consequence: provincial data inherits all uncertainty from national estimates plus 
additional uncertainty from the averaging process. Granular province-level comparisons 
should be interpreted with caution.